In [1]:
import requests
import json

In [2]:
# # 服务器地址
BASE_URL = "http://127.0.0.1:8020"

### 健康检查

In [3]:
# 1. 健康检查示例
def check_health():
    url = f"{BASE_URL}/health"
    response = requests.get(url)
    return response.json()

health_result = check_health()
print("Health check:", health_result)

Health check: {'status': 'healthy'}


### 插入文本

In [5]:
# 2. 插入文本示例
def insert_text(text):
    url = f"{BASE_URL}/insert"
    headers = {"Content-Type": "application/json"}
    data = {
        "text": text
    }
    response = requests.post(url, json=data, headers=headers)
    return response.json()

text = "我的名字叫无师大模型"
insert_result = insert_text(text)
print("Insert result:", insert_result)

Insert result: {'status': 'success', 'data': None, 'message': 'Text inserted successfully'}


### 插入文件

In [6]:
# 3. 插入文件示例
def insert_file(file_path):
    url = f"{BASE_URL}/insert_file"
    with open(file_path, 'rb') as file:
        files = {'file': file}
        response = requests.post(url, files=files)
    return response.json()

file_path = '../resources/raw_docs/xiaolu.txt'
file_result = insert_file(file_path)
print("File insert result:", file_result)

File insert result: {'status': 'success', 'data': None, 'message': '成功处理文件 xiaolu.txt 并插入内容'}


In [7]:
# excel 文件
file_path = '../resources/raw_docs/XiaoMing-Life.xlsx'
file_result = insert_file(file_path)
print("File insert result:", file_result)

File insert result: {'status': 'success', 'data': None, 'message': '成功处理文件 XiaoMing-Life.xlsx 并插入内容'}


### 查询示例

In [8]:
# 4. 查询示例
def make_query(query_text, mode="hybrid", only_need_context:bool=False):
    url = f"{BASE_URL}/query/full"
    headers = {"Content-Type": "application/json"}
    data = {
        "query": query_text,
        "mode": mode,
        "only_need_context": only_need_context
    }
    response = requests.post(url, json=data, headers=headers)
    return json.dumps(response.json(), indent=4, ensure_ascii=False)

# query = "你是谁"
# query = "小陆是谁"
# query = "小明是谁"
# query = "小明和小陆有什么关系"
query = "小明的父亲是谁"
mode = ["naive","local","global","hybrid"][0] 
only_need_context = False
query_result = make_query(query, mode, only_need_context)
print("Query result:", query_result)

print('-'*42)
print(json.loads(query_result)['data'])

Query result: {
    "status": "success",
    "data": "<think>\n嗯，用户问的是“小明的父亲是谁”。我需要根据提供的文档 chunks 来回答这个问题。首先，我浏览了所有相关 chunk，发现里面只有 K001 和 K002 提到了小明的情况。\n\n在 K001 中，小明出生于 2000 年 5 月 1 日，来自上海。K002 则提到小明小时候喜欢玩积木和看动画片。这些 chunk 都没有涉及到小明的父亲是谁的信息。\n\n根据响应规则，如果提供的文档 chunks 没有相关信息，我不能随便编造答案，否则可能会误导用户。因此，在这种情况下，我应该礼貌地告诉用户，找不到相关的信息。\n</think>\n\n目前提供的信息中并没有提到小明的父亲是谁。",
    "message": null
}
------------------------------------------
<think>
嗯，用户问的是“小明的父亲是谁”。我需要根据提供的文档 chunks 来回答这个问题。首先，我浏览了所有相关 chunk，发现里面只有 K001 和 K002 提到了小明的情况。

在 K001 中，小明出生于 2000 年 5 月 1 日，来自上海。K002 则提到小明小时候喜欢玩积木和看动画片。这些 chunk 都没有涉及到小明的父亲是谁的信息。

根据响应规则，如果提供的文档 chunks 没有相关信息，我不能随便编造答案，否则可能会误导用户。因此，在这种情况下，我应该礼貌地告诉用户，找不到相关的信息。
</think>

目前提供的信息中并没有提到小明的父亲是谁。


### 流式查询

In [9]:
import requests
from IPython.display import display, clear_output

# 查询参数
query_data = {
    "query": "小明的父亲是谁",
    "mode": "hybrid",
    "only_need_context": False
}
url = BASE_URL + "/query/stream"

# 流式客户端
def stream_client():
    # 创建一个显示区域用于实时更新
    output_area = display("", display_id=True)
    
    try:
        # 发送POST请求，启用流式响应
        response = requests.post(
            url,
            json=query_data,
            stream=True  # 启用流式响应
        )
        
        # 检查响应状态
        if response.status_code != 200:
            print(f"Error: Received status code {response.status_code}")
            return
        
        # 逐块读取流式内容
        for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
            if chunk:  # 过滤空块
                # 直接处理原始内容
                if chunk.startswith('ERROR:'):
                    print(f"\nError: {chunk[6:]}")
                    break
                else:
                    # 实时更新显示
                    # clear_output(wait=True)
                    output_area.update(f"Received: {chunk}")
                    print(chunk, end='', flush=True)
        
        response.close()
        # print("\nStream completed")
        
    except Exception as e:
        print(f"Client error: {str(e)}")

# 执行客户端
stream_client()

''

<think>
好吧，用户问的是“小明的父亲是谁”。首先，我需要从提供的知识库中查找相关信息。查看实体部分，发现有关于“小明”的记录，提到他出生于2000年5月1日，在上海上学，主修计算机科学。但并没有提到关于家庭成员的信息，比如父母是谁或者他们的背景。

接着，看看关系部分是否有相关的信息。知识库中有“家庭记录”相关的条目K001和K004，但这些内容主要涉及小明的出生地点、学校以及他读大学的专业，并没有涉及他的家庭结构或父亲的情况。因此，我没有找到关于小明父亲身份的具体信息。

根据规则，我只能基于提供的知识库内容回答问题，而无法添加额外的信息。所以，我的回答是无法确定小明的父亲是谁。
</think>

根据提供的知识库，小明的个人生活部分提到了他的婚姻情况和结婚时间，但没有涉及他的家庭成员或父亲的身份。因此，无法确定小明的父亲是谁。

如果您有其他问题，欢迎继续提问！